# A minimal Duffing SINN example

This example learns the two position observables of the coupled Duffing benchmark at $\lambda=4$ using one SINN training run. It reuses the original data loaders, network, statistical objective, checkpoint selection, and evaluator in `duffing/`.

Set `MODE="quick"` for a 60-iteration installation/workflow check, or `MODE="full"` for a 10,000-iteration training demonstration. Both modes use the supplied target archive and the same model, losses, and ensemble sizes. A single run is not the paper's across-run study. An optional setting regenerates the data in a separate directory.

Run the cells from top to bottom. Every **Restart Kernel and Run All** creates a new output directory; the supplied paper data and results are never overwritten.


## 1. Environment and configuration

Use a Python environment containing NumPy, Matplotlib, PyTorch, and Jupyter. The README gives setup and headless execution commands. On a cluster, run training on an allocated compute node, not a login node.

CUDA is selected when available; otherwise this example uses CPU. Results can differ across devices and software versions.


In [ ]:
import importlib.util
import json
import os
import platform
import shlex
import subprocess
import sys
import tempfile
import time
from pathlib import Path

# Bound CPU thread use, respecting a smaller Slurm CPU allocation.
CPU_THREADS = max(1, min(4, int(os.environ.get("SLURM_CPUS_PER_TASK") or 4)))
for variable in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ[variable] = str(CPU_THREADS)

required = ("numpy", "matplotlib", "torch")
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        "Missing packages in this notebook kernel: " + ", ".join(missing)
        + ". Install them in the kernel's Python environment; see README.md."
    )

import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython import get_ipython
from IPython.display import Markdown, display

shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")

candidates = (Path.cwd(), *Path.cwd().parents)
REPO_ROOT = next(
    (p for p in candidates if (p / "duffing" / "train_sinn.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the cloned repository.")
DUFFING = REPO_ROOT / "duffing"
sys.path.insert(0, str(DUFFING))
import evaluate as EV

if Path(EV.__file__).resolve() != (DUFFING / "evaluate.py").resolve():
    raise RuntimeError("A different evaluate module is loaded. Restart the kernel.")

print("Repository:", REPO_ROOT)
print("Python:", sys.version.split()[0], "| NumPy:", np.__version__)
print("PyTorch:", torch.__version__, "| Matplotlib:", plt.matplotlib.__version__)


In [ ]:
# Change only MODE to switch between the two training budgets.
MODE = "quick"              # "quick": 60 iterations; "full": 10,000.
REGENERATE_TARGET = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0

MODE_SETTINGS = {
    "quick": (60, 20),
    "full": (10_000, 200),
}
if MODE not in MODE_SETTINGS:
    raise ValueError("MODE must be 'quick' or 'full'.")
TRAIN_ITERS, VAL_EVERY = MODE_SETTINGS[MODE]

# Keep these settings consistent with the original Duffing experiment.
LAMBDA = 4
LAYERS, HIDDEN = 2, 10
N_F = 3                       # Three oblique frames plus the coordinate frame.
D_IN = 2                      # Two active channels; network input width is four.
N_TRAIN = N_EVAL = 400
WARMUP = 200                   # Extra recurrent steps, discarded before scoring.
TAU_SQUARED = 10.0
TAU_CLI = TAU_SQUARED ** 0.5    # Original CLI uses unsquared error ratios.

if TRAIN_ITERS < 1 or VAL_EVERY < 1:
    raise ValueError("TRAIN_ITERS and VAL_EVERY must be positive.")
if DEVICE not in ("cpu", "cuda"):
    raise ValueError("Select 'cpu' or 'cuda'.")
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA was requested but is unavailable in this kernel.")

CONFIG = dict(
    mode=MODE, lambda_value=LAMBDA, seed=SEED, layers=LAYERS, hidden=HIDDEN,
    noncoordinate_frames=N_F, total_frames=N_F + 1, d_in=D_IN,
    n_train=N_TRAIN, n_eval=N_EVAL, warmup=WARMUP,
    iterations=TRAIN_ITERS, validation_every=VAL_EVERY,
    loss_balancing=True, tau_squared=TAU_SQUARED,
    regenerate_target=REGENERATE_TARGET, device=DEVICE, cpu_threads=CPU_THREADS,
)
print(json.dumps(CONFIG, indent=2))
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
EXAMPLE_DIR = REPO_ROOT / "examples" / "duffing"
OUTPUT_BASE = EXAMPLE_DIR / "outputs"
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
RUN_DIR = Path(tempfile.mkdtemp(prefix=f"demo-{MODE}-", dir=OUTPUT_BASE))
TRAIN_DIR, REPORT_DIR, LOG_DIR = (
    RUN_DIR / "training", RUN_DIR / "evaluation", RUN_DIR / "logs"
)
for folder in (TRAIN_DIR, REPORT_DIR, LOG_DIR):
    folder.mkdir()
STAGE_SECONDS = {}

def run_stage(script_name, *arguments):
    """Run an unchanged Duffing entry point and preserve its console log."""
    script = DUFFING / script_name
    if not script.is_file():
        raise FileNotFoundError(script)
    command = [sys.executable, "-u", str(script), *map(str, arguments)]
    log_path = LOG_DIR / (script.stem + ".log")
    print("$", shlex.join(command))
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        with subprocess.Popen(
            command, cwd=DUFFING, env=os.environ.copy(),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        ) as process:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
            return_code = process.wait()
    STAGE_SECONDS[script.stem] = time.perf_counter() - started
    if return_code:
        raise RuntimeError(
            f"{script_name} exited with code {return_code}; inspect {log_path}"
        )
    print(f"\nFinished {script_name} in {STAGE_SECONDS[script.stem]:.1f} s")

(RUN_DIR / "demo_config.json").write_text(
    json.dumps(CONFIG, indent=2), encoding="utf-8"
)
print("All new files go to:", RUN_DIR)


## 2. Target data and training statistics

By default, the notebook uses the supplied 20,000-trajectory archive at $\lambda=4$, with 200 saved states and two position components per trajectory. The original reference file fixes the fitting, reference-block, validation, and evaluation splits, standardization, density bandwidths, and projection scales.

Set `REGENERATE_TARGET=True` above and rerun from the top to generate a new archive, rebuild the reference, and compute fitting statistics. This optional branch uses 20,000 trajectories and writes only inside the new example output directory. It takes additional computation; no new simulation-convergence study is performed here.


In [ ]:
if REGENERATE_TARGET:
    DATA_DIR = RUN_DIR / "data"
    DATA_DIR.mkdir()
    run_stage(
        "make_target.py", "--outdir", DATA_DIR,
        "--lambdas", LAMBDA, "--ntraj", 20000,
    )
    run_stage("make_reference.py", "--datadir", DATA_DIR)
    run_stage("make_target_stats.py", "--datadir", DATA_DIR)
else:
    DATA_DIR = DUFFING / "data"
    print("Reusing the included target archive and statistics (read-only).")

TAG = f"lam{LAMBDA:g}".replace(".", "p")
needed = (
    f"traj_{TAG}.npy", f"reference_{TAG}.npz", f"target_stats_{TAG}.npz"
)
for filename in needed:
    if not (DATA_DIR / filename).is_file():
        raise FileNotFoundError(DATA_DIR / filename)

target_path = DATA_DIR / f"traj_{TAG}.npy"
ref = EV.load_reference(str(DATA_DIR), TAG, str(target_path))
target = np.load(target_path, mmap_mode="r")
split = EV.split_from_reference(ref)
if int(ref["b_fit"]) != N_TRAIN or int(ref["l_seq"]) != 200:
    raise ValueError("The included reference must use N=400 and T=200.")
print("Target shape (time, trajectory, component):", target.shape)
print("Trajectory-index splits (zero-based, half-open):")
for name, (lo, hi) in split.items():
    print(f"  {name:10s}: [{lo}, {hi}) -> {hi - lo} trajectories")
print("Reference binding:", ref["constants_sha256"][:16])


In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 2.6))
sample_index = np.arange(target.shape[0])
for component, color in ((0, "tab:blue"), (1, "tab:orange")):
    ax.plot(
        sample_index, target[:, 0, component], color=color, lw=1.2,
        label=rf"$q_{component + 1}$",
    )
ax.set(xlabel="Saved sample index", ylabel="Position")
ax.set_title("One target trajectory (illustration only)", fontsize=10)
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
fig.savefig(RUN_DIR / "target_trajectory.png", dpi=150)
plt.show()


## 3. Train one SINN model

The model has two LSTM layers of width 10, two active Gaussian input channels, and adaptive balancing of five statistical families: ACF, CCF, ACF2, PDF, and PROJ. Each iteration generates 400 trajectories, discards 200 recurrent warm-up steps, and scores the following 200 steps.

Quick mode uses 60 iterations and validation every 20 iterations; full mode uses 10,000 iterations and validation every 200 iterations. Only these two settings change between modes. The retained checkpoint minimizes the unweighted validation aggregate.

The training and validation curves below compare different target statistics: training uses cached fitting targets; validation uses held-out correlations and kernel-smoothed analytic PDF/PROJ targets. Neither curve is required to decrease monotonically.


In [ ]:
run_stage(
    "train_sinn.py",
    "--datadir", DATA_DIR, "--outdir", TRAIN_DIR,
    "--lam", LAMBDA, "--nf", N_F, "--lb", "on", "--din", D_IN,
    "--layers", LAYERS, "--hidden", HIDDEN, "--seed", SEED,
    "--iters", TRAIN_ITERS, "--val-every", VAL_EVERY,
    "--batch", N_TRAIN, "--n-gen", N_EVAL, "--warmup", WARMUP,
    "--lr-theta", 1e-3, "--lr-sigma", 1e-3,
    "--tau", TAU_CLI, "--device", DEVICE,
)

run_tag = (
    f"nf{N_F}_lbon_din{D_IN}_L{LAYERS}H{HIDDEN}_{TAG}_s{SEED:02d}"
)
record = json.loads((TRAIN_DIR / f"run_{run_tag}.json").read_text())
with np.load(TRAIN_DIR / f"hist_{run_tag}.npz", allow_pickle=False) as saved:
    history = {key: saved[key].copy()
               for key in ("iter", "train_unweighted", "val_agg")}
if not all(np.isfinite(values).all() for values in history.values()):
    raise RuntimeError("Training or validation history contains non-finite values.")
checkpoint = TRAIN_DIR / f"net_{run_tag}.pt"
if not checkpoint.is_file():
    raise RuntimeError("Training did not retain a checkpoint.")
validation_ratio = float(record["best"]["worst_ratio"]) ** 2
print("Retained iteration:", record["best"]["iter"])
print("Retained validation aggregate:", record["best"]["val_agg"])
print("Validation worst-family loss ratio (squared):", validation_ratio)
print(
    "Validation-successful at the paper threshold:",
    validation_ratio <= TAU_SQUARED,
)
if MODE == "quick":
    print("Quick mode checks execution; it need not pass the accuracy threshold.")
else:
    print("This is one training run, not the paper's across-run study.")


In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 3.0))
ax.semilogy(
    history["iter"], history["train_unweighted"], "o-",
    label="Training sum (fitting targets)",
)
ax.semilogy(
    history["iter"], history["val_agg"], "s--",
    label="Validation sum (selection score)",
)
ax.axvline(record["best"]["iter"], color="0.65", ls=":", lw=1)
ax.set(xlabel="Iteration", ylabel="Sum of squared discrepancies")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(RUN_DIR / "training_history.png", dpi=150)
plt.show()


## 4. Evaluate the retained checkpoint

The original evaluator generates a new 400-trajectory ensemble using its evaluation noise stream. Correlations are compared with the 12,000 held-out target trajectories; PDF and PROJ use the kernel-smoothed analytic target. Each reference loss comes from comparisons of the nine 400-trajectory reference blocks with that same target.

The archived reporting CLI prints unsquared error ratios. The table below squares those ratios once to display $r_s^{\mathrm{eval}}=\mathcal L_s^{\mathrm{eval}}/\mathcal L_s^{\mathrm{ref}}$. This evaluation does not change checkpoint selection or the validation outcome.


In [ ]:
run_stage(
    "report.py",
    "--datadir", DATA_DIR, "--rundir", TRAIN_DIR, "--outdir", REPORT_DIR,
    "--lam", LAMBDA, "--nf", N_F, "--lb", "on", "--din", D_IN,
    "--layers", LAYERS, "--hidden", HIDDEN,
    "--warmup", WARMUP, "--n-gen", N_EVAL,
    "--tau", TAU_CLI, "--device", DEVICE,
    "--against", "evaluation", "--save-display",
)

config_tag = f"nf{N_F}_lbon_din{D_IN}_L{LAYERS}H{HIDDEN}_{TAG}"
report = json.loads((REPORT_DIR / f"report_{config_tag}.json").read_text())
if report["n_seeds"] != 1 or report["n_gen"] != N_EVAL:
    raise RuntimeError("Expected one newly trained model and 400 evaluation paths.")
if report["anchor"] != "evaluation":
    raise RuntimeError("Expected the held-out evaluation target.")
families = ("ACF", "CCF", "ACF2", "PDF", "PROJ")
losses = {name: float(report["table"][name]["e_mod"]) ** 2
          for name in families}
ratios = {name: float(report["table"][name]["ratio"]) ** 2
          for name in families}
if not all(np.isfinite(value) for value in (*losses.values(), *ratios.values())):
    raise RuntimeError("Evaluation returned non-finite family discrepancies.")

lines = [
    "| Family | Evaluation loss (squared) | Reference-normalized loss ratio |",
    "| :--- | ---: | ---: |",
]
lines += [f"| {name} | {losses[name]:.5g} | {ratios[name]:.5g} |"
          for name in families]
display(Markdown("\n".join(lines)))


## 5. Inspect the learned statistics

Black solid and red dashed curves denote target and SINN, respectively. Marginal-density targets are the kernel-smoothed analytic densities in standardized coordinates. ACF targets are estimated from the held-out target trajectories.

The ACFs use the original loss estimator with $NT$ normalization, not a pair-count correction. These single-run curves are not replacements for the paper's across-run results.


In [ ]:
generated_path = REPORT_DIR / f"gen_{report['display_run']}.npz"
generated, coordinate_space = EV.load_generated(
    str(generated_path), ref, "standardized"
)
if generated.shape != (200, N_EVAL, 2):
    raise RuntimeError(f"Unexpected generated shape: {generated.shape}")
EV.check_generated(generated, target, ref, coordinate_space)

generated_stats = EV.stats(generated, ref, coordinate_space)
lo, hi = split["evaluation"]
target_standardized = EV.standardise(target[:, lo:hi, :], ref, "raw")
target_acf, _, _ = EV.correlations(target_standardized, int(ref["k_lags"]))
z_grid = ref["z_grid"]
lags = np.arange(target_acf.shape[0])

fig, axes = plt.subplots(2, 2, figsize=(7.2, 4.7))
for component in (0, 1):
    ax = axes[0, component]
    ax.plot(z_grid, ref["marginal"][:, component], "k-", lw=1.5, label="Target")
    ax.plot(z_grid, generated_stats["PDF"][:, component],
            color="red", ls="--", lw=1.5, label="SINN")
    ax.set(xlabel=rf"$\widetilde{{q}}_{component + 1}$", ylabel="Density")
    ax.set_title(f"Component {component + 1} marginal", fontsize=10)

    ax = axes[1, component]
    ax.plot(lags, target_acf[:, component], "k-", lw=1.5)
    ax.plot(lags, generated_stats["ACF"][:, component],
            color="red", ls="--", lw=1.5)
    ax.axhline(0, color="0.8", lw=0.7)
    ax.set(xlabel="Lag k", ylabel="ACF")
    ax.set_title(f"Component {component + 1} ACF", fontsize=10)

axes[0, 0].legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(RUN_DIR / "statistic_comparison.png", dpi=150)
plt.show()


### Joint-density contours

The black solid contours show the kernel-smoothed analytic target, and the red dashed contours show the joint KDE of the same 400 SINN trajectories used above. No additional trajectories are generated. Joint density is a diagnostic, not a training or checkpoint-selection loss.

Contour levels enclose 10%, 30%, 50%, 70%, and 90% of the target mass. These target-derived density levels are reused for SINN; they are not recalculated from the SINN density. Axes are in standardized coordinates. The plot is saved as PNG and PDF alongside its numerical density arrays.


In [ ]:
sys.path.insert(0, str(EXAMPLE_DIR))
from plot_joint import plot_joint_density

fig, contour_levels = plot_joint_density(
    z_grid, ref["joint"], generated_stats["JOINT"], RUN_DIR
)
plt.show()
print("Joint-density contours:", RUN_DIR / "joint_density_contours.pdf")


## 6. Saved outputs

The new output directory contains the checkpoint and training history, the evaluator's report and generated ensemble, console logs, four PNG figures, the joint-density PDF and arrays, and a compact summary recording the selected mode. Completing quick mode checks the pipeline, not the paper's accuracy criterion.

After changing `MODE` or other settings, restart the kernel and run all cells to create a fresh output directory rather than rerunning only the training cell. To add the new contour to an already completed run without retraining, see the plotting-only command in the README.


In [ ]:
summary = {
    "purpose": "one-run workflow demonstration, not a paper reproduction",
    "config": CONFIG,
    "environment": {
        "python": platform.python_version(), "numpy": np.__version__,
        "torch": str(torch.__version__),
        "matplotlib": plt.matplotlib.__version__, "device": DEVICE,
        "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
    },
    "constants_sha256": ref["constants_sha256"],
    "target_sha256": ref["target_sha256"],
    "retained_iteration": int(record["best"]["iter"]),
    "validation_worst_family_squared_ratio": validation_ratio,
    "validation_successful": bool(validation_ratio <= TAU_SQUARED),
    "evaluation_loss_squared": losses,
    "evaluation_reference_normalized_squared_ratio": ratios,
    "generated_shape": list(generated.shape),
    "all_generated_states_finite": bool(np.isfinite(generated).all()),
    "stage_seconds": STAGE_SECONDS,
}
summary_path = RUN_DIR / "demo_summary.json"
summary_path.write_text(json.dumps(summary, indent=2, allow_nan=False), encoding="utf-8")
print("Completed demonstration:", RUN_DIR)
print("Summary:", summary_path)
print("Total subprocess time:", round(sum(STAGE_SECONDS.values()), 1), "s")
